# Predictive Maintenance - Model Training

This notebook demonstrates how to train models for RUL prediction.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from predictive_maintenance.models.classical_models import RULPredictor
from predictive_maintenance.features.feature_extraction import SensorFeatureExtractor, RULFeatureEngineer

## 1. Load Data

In [ ]:
# Load train and test data
train_df = pd.read_csv('../data/train_data.csv')
test_df = pd.read_csv('../data/test_data.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

## 2. Feature Engineering

In [ ]:
# Select sensor columns
sensor_cols = [col for col in train_df.columns if col.startswith('sensor_')]

# Extract features
feature_engineer = RULFeatureEngineer()
train_features = feature_engineer.create_degradation_features(train_df, sensor_cols)
test_features = feature_engineer.create_degradation_features(test_df, sensor_cols)

print(f"Features created: {train_features.shape}")

## 3. Prepare Training Data

In [ ]:
# Select feature columns (excluding metadata and target)
exclude_cols = ['unit_id', 'cycle', 'timestamp', 'RUL', 'will_fail']
feature_cols = [col for col in train_features.columns if col not in exclude_cols]

# Prepare training data
X_train = train_features[feature_cols].fillna(0)
y_train = train_features['RUL']

X_test = test_features[feature_cols].fillna(0)
y_test = test_features['RUL']

print(f"Training features: {X_train.shape}")
print(f"Test features: {X_test.shape}")

## 4. Train Random Forest Model

In [ ]:
# Initialize and train model
rf_model = RULPredictor(model_type='random_forest')
rf_model.build_model(n_estimators=100, max_depth=10, random_state=42)
rf_model.train(X_train, y_train)

print("Random Forest model trained!")

## 5. Evaluate Model

In [ ]:
# Make predictions
y_pred = rf_model.predict(X_test)

# Calculate metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R²: {r2:.4f}")

## 6. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

# Get feature importance
importance = rf_model.get_feature_importance()
importance_df = pd.DataFrame({
    'feature': list(importance.keys()),
    'importance': list(importance.values())
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Save Model

In [ ]:
# Save model
rf_model.save_model('../models/rul_predictor_rf.pkl')
print("Model saved successfully!")

## 8. Train XGBoost Model (Optional)

In [ ]:
# Initialize and train XGBoost model
xgb_model = RULPredictor(model_type='xgboost')
xgb_model.build_model(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
xgb_model.train(X_train, y_train)

# Evaluate
y_pred_xgb = xgb_model.predict(X_test)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)

print(f"XGBoost RMSE: {rmse_xgb:.2f}")
print(f"XGBoost MAE: {mae_xgb:.2f}")
print(f"XGBoost R²: {r2_xgb:.4f}")

# Save XGBoost model
xgb_model.save_model('../models/rul_predictor_xgb.pkl')